<a href="https://colab.research.google.com/github/mr-zero-000/Statistical-Learning-e23034/blob/main/Assignment%209/Question_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates**

# **Task 1: Prior Belief Boundaries**

In [1]:
# @title Execute this cell to generate the Prior Beta PDF visualization
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Define the physical domain matching the boundaries
theta_vals = np.linspace(0.01, 1.0, 500)

# Prior parameters: Beta(8, 1.5)
alpha_0, beta_0 = 8.0, 1.5
prior_density = beta.pdf(theta_vals, alpha_0, beta_0)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=theta_vals, y=prior_density,
    mode='lines',
    name='Initial Prior: Beta(8, 1.5)',
    line=dict(color='darkcyan', width=3)
))

fig.update_layout(
    title="Initial Prior Belief Boundaries: Beta(8, 1.5)",
    xaxis_title="Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density f(θ)",
    xaxis=dict(range=[0, 1.05]),
    template="plotly_white",
    hovermode="x unified"
)
fig.show()

# **2. Analytical Calculation of Prior Mean**

The expected value of a standard Beta distribution parameterized by $\alpha_0$ and $\beta_0$ over the domain $[0, 1]$ is computed analytically as follows:$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha_0}{\alpha_0 + \beta_0} = \frac{8}{8 + 1.5} = \frac{8}{9.5} = \frac{16}{19} \approx 0.8421$$

# **3. Engineering Justification for the Choice of Prior**

This specific $\text{Beta}(8, 1.5)$ distribution serves as an ideal structural prior for several reasons:Domain Matching: The support of the Beta distribution is bounded exactly on the interval $[0, 1]$, which physically mirrors the structural remaining stiffness efficiency factor where negative values or values exceeding $1.0$ are physically impossible.Optimistic Structural Baseline: Because $\alpha_0 = 8 \gg \beta_0 = 1.5$, the center of mass is heavily skewed rightward toward $1.0$. This mathematically embeds the real-world operational assumption that an in-service aircraft wing or bridge girder is highly likely to be healthy and pristine at deployment, while still leaving a small non-zero tail for pre-existing flaws or transportation defects.

# **Task 2: Structural Likelihood Formulation**

1. Mathematical Derivation of the Isolated Likelihood ComponentThe physical equation relating the continuous measurement $y_k$ to the latent stiffness factor $\theta$ is:$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$To derive the likelihood distribution $L(y_k \mid \theta) = f_{Y_k \mid \Theta}(y_k \mid \theta)$, we perform a change of variables or use the properties of the log-normal distribution. Let us rearrange the physical model to isolate the normal error term $\epsilon_k$:$$\frac{y_k}{\theta \cdot K_{\text{nominal}}} = e^{\epsilon_k}$$Taking the natural logarithm of both sides:$$\ln\left(\frac{y_k}{\theta \cdot K_{\text{nominal}}}\right) = \epsilon_k$$Using the logarithmic identity $\ln(A/B) = \ln(A) - \ln(B)$:$$\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}) = \epsilon_k$$Since $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$, the transformed variable $W_k = \ln(Y_k)$ follows a normal distribution:$$W_k = \ln(Y_k) \sim \mathscr{N}\left(\ln(\theta \cdot K_{\text{nominal}}), \, \sigma^2\right)$$By definition, if the natural logarithm of a random variable is normally distributed, the variable itself follows a log-normal distribution. The probability density function of a log-normal variable $Y_k$ is:$$f_{Y_k}(y_k) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{(\ln(y_k) - \mu)^2}{2\sigma^2} \right)$$Substituting the parameters $\mu = \ln(\theta \cdot K_{\text{nominal}})$, we obtain the isolated likelihood contribution of a single sensor observation:$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left[\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})\right]^2}{2\sigma^2} \right) \quad \re$$2. Joint Likelihood Function FormulationAssuming the sensor readings are conditionally independent at each inspection step given the true parameter $\theta$, the joint likelihood for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of their individual log-normal likelihood components:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} L(y_i \mid \theta) = \prod_{i=1}^{k} \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( -\frac{\left[\ln(y_i) - \ln(\theta \cdot K_{\text{nominal}})\right]^2}{2\sigma^2} \right)$$Using exponent laws to turn the product into a summation within the exponential component:$$L(\mathbf{y}^{(k)} \mid \theta) = \left( \frac{1}{\sigma \sqrt{2\pi}} \right)^k \left( \prod_{i=1}^{k} \frac{1}{y_i} \right) \exp\left( -\frac{1}{2\sigma^2} \sum_{i=1}^{k} \left[\ln(y_i) - \ln(\theta \cdot K_{\text{nominal}})\right]^2 \right)$$

# **Task 3: Mathematical Formulation of the Non-Conjugate Grid Update**

. Explanation of Non-ConjugacyAn exact closed-form analytical solution for the posterior density does not exist here because the Beta prior distribution and the log-normal measurement likelihood form a non-conjugate pair.The Beta prior kernel consists of algebraic power bases ($\theta^{\alpha-1}(1-\theta)^{\beta-1}$), whereas the structural likelihood kernel contains $\theta$ embedded inside a natural logarithm which is itself squared within an exponential function ($\exp(-[\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})]^2)$). When multiplied together, these two functional forms cannot be simplified into a kernel matching any known, standard probability density function family. The normalizer integral cannot be solved analytically, necessitating a numerical grid approximation.2. Recursive Posterior Proportionality RelationshipBy applying Bayes' Theorem sequentially, the posterior distribution at the current step $k$ is proportional to the product of the newest isolated likelihood contribution and the previous step's posterior density:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$Substituting the structural likelihood derived in Task 2:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left[\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})\right]^2}{2\sigma^2} \right) \right] \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \quad \re$$

# **Task 4: Running Point Estimates**

Because we cannot compute these metrics via closed-form solutions, we extract point estimators by defining definite integral equations across the bounded domain $(0, 1]$:1. Running Posterior Mean (Expected A Posteriori - EAP)The posterior mean represents the minimum mean squared error (MMSE) estimator:$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \frac{\int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta}{\int_{0}^{1} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta}$$If the density function is already normalized such that the denominator integrates to $1$, the equation simplifies to:$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$2. Running Maximum A Posteriori (MAP)The MAP estimate corresponds to the mode of the posterior distribution—the value of $\theta$ that maximizes the posterior density over the bounded physical parameter space:$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

# **Task 5: Algorithmic Grid Approximation and Normalization**

1.  **Grid Initialization**: Construct a fixed evaluation mesh vector $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$ consisting of $M$ equally spaced nodes spanning from $\theta_1 = 0.01$ to $\theta_M = 1.0$. The step size is defined as $\Delta \theta = \frac{1.0 - 0.01}{M - 1}$. Compute the initial prior density vector $\mathbf{f}^{(0)}$ by evaluating the $\text{Beta}(8, 1.5)$ distribution pointwise at each grid node $\theta_m$.
2.  **Handle Boundary Limits**: We physically cap the grid boundaries at $0.01$ to avoid evaluation errors, as $\ln(\theta)$ approaches $-\infty$ when $\theta \to 0$.
3.  **Pointwise Likelihood Multiplication**: Upon receiving a new continuous sensor measurement $y_k$, compute the unnormalized likelihood value at every mesh node. Update the unnormalized posterior vector $\mathbf{f}_{\text{unnorm}}^{(k)}$ via pointwise multiplication:
	$$f_{\text{unnorm}}^{(k)}(\theta_m) = \exp\left( -\frac{\left[\ln(y_k) - \ln(\theta_m \cdot K_{\text{nominal}})\right]^2}{2\sigma^2} \right) \cdot f^{(k-1)}(\theta_m)$$
	(Note: The scaling constant $\frac{1}{y_k \sigma \sqrt{2\pi}}$ can be safely ignored during this step because it cancels out during normalization).
4.  **Sequential Normalization via Trapezoidal Rule**: To find the total mass under the unnormalized posterior curve, apply the composite trapezoidal rule:
	$$\text{Mass} = \frac{\Delta \theta}{2} \left[ f_{\text{unnorm}}^{(k)}(\theta_1) + 2\sum_{m=2}^{M-1} f_{\text{unnorm}}^{(k)}(\theta_m) + f_{\text{unnorm}}^{(k)}(\theta_M) \right]$$
	Divide each unnormalized element by this integrated scalar value to compute the fully normalized running posterior distribution:
	$$\mathbf{f}^{(k)} = \frac{\mathbf{f}_{\text{unnorm}}^{(k)}}{\text{Mass}}$$

# **Task 6: Performance Tracking and Degradation Convergence Analysis**

In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import beta

# Set seed for reproducible simulation results
np.random.seed(42)

# --- Configuration Constants ---
theta_true = 0.68
k_nominal = 50.0  # kN/mm
sigma = 0.15
n_measurements = 15

# --- Grid Resolution Initialization ---
grid_size = 1000
theta_grid = np.linspace(0.01, 1.0, grid_size)

# Calculate the initial prior vector: Beta(8, 1.5)
prior_density = beta.pdf(theta_grid, 8, 1.5)
# Normalize using the trapezoidal rule
current_density = prior_density / np.trapezoid(prior_density, theta_grid)

# --- Tracking Arrays ---
milestones = [0, 1, 2, 5, 10, 15]
saved_densities = {0: current_density.copy()}

# Prior tracking values at step 0
history_bayes_mean = [np.trapezoid(theta_grid * current_density, theta_grid)]
history_map_est = [theta_grid[np.argmax(current_density)]]
steps = [0]

# --- Sequential Non-Conjugate Update Timeline Loop ---
for k in range(1, n_measurements + 1):
    # 1. Simulate noisy log-normal sensor reading from degradation physics
    epsilon_k = np.random.normal(0, sigma)
    y_k = theta_true * k_nominal * np.exp(epsilon_k)

    # 2. Compute likelihood values across the grid (ignoring constants)
    likelihood = np.exp(-((np.log(y_k) - np.log(theta_grid * k_nominal)) ** 2) / (2 * sigma ** 2))

    # 3. Recursive Multiplication Step
    unnorm_posterior = likelihood * current_density

    # 4. Sequential Normalization via Trapezoidal Rule
    current_density = unnorm_posterior / np.trapezoid(unnorm_posterior, theta_grid)

    # 5. Extract Running Point Estimators
    bayes_mean = np.trapezoid(theta_grid * current_density, theta_grid)
    map_est = theta_grid[np.argmax(current_density)]

    # Store records
    if k in milestones:
        saved_densities[k] = current_density.copy()

    history_bayes_mean.append(bayes_mean)
    history_map_est.append(map_est)
    steps.append(k)

# --- Visualization 1: Posterior Density Curves at Milestones ---
fig1 = go.Figure()
colors = {0: 'gray', 1: 'purple', 2: 'blue', 5: 'orange', 10: 'green', 15: 'darkred'}

for m in milestones:
    dash_style = 'dash' if m == 0 else 'solid'
    fig1.add_trace(go.Scatter(
        x=theta_grid, y=saved_densities[m],
        mode='lines',
        name=f'Step k = {m}',
        line=dict(color=colors[m], width=2.5, dash=dash_style)
    ))

fig1.update_layout(
    title="Evolution of the Bounded Posterior Grid Density over Milestones",
    xaxis_title="Stiffness Efficiency Factor (θ)",
    yaxis_title="Normalized Probability Density",
    template="plotly_white",
    hovermode="x unified"
)
fig1.show()

# --- Visualization 2: Convergence Tracking Timeline ---
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=steps, y=history_bayes_mean,
    mode='lines+markers',
    name='Posterior Mean (EAP)',
    line=dict(color='darkblue', width=2),
    marker=dict(size=6)
))

fig2.add_trace(go.Scatter(
    x=steps, y=history_map_est,
    mode='lines+markers',
    name='Maximum A Posteriori (MAP)',
    line=dict(color='darkorange', width=2, dash='dash'),
    marker=dict(size=6)
))

fig2.add_trace(go.Scatter(
    x=steps, y=[theta_true] * len(steps),
    mode='lines',
    name=f'True Stiffness (θ = {theta_true})',
    line=dict(color='crimson', width=1.5, dash='dot')
))

fig2.update_layout(
    title="Convergence of Non-Conjugate Bounded Estimators against True Parameter",
    xaxis_title="Inspection Step (Sensor Reading k)",
    yaxis_title="Estimated Efficiency Factor (θ)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=1),
    template="plotly_white",
    hovermode="x unified"
)
fig2.show()

# **Convergence Analysis**

1.  **Overcoming the Optimistic Prior**

The system typically overcomes the optimistic prior ($\mathbb{E}[\Theta^{(0)}] \approx 0.842$) within the first 2 to 3 sensor readings. Because the structural measurement noise ($\sigma = 0.15$) is reasonably low, the log-normal likelihood contribution acts as a powerful evidence collector. A few continuous sensor readings indicating lower values quickly suppress the probability mass at the higher end of the scale, driving the distribution down toward the true degradation state.

2.  **Structural Safety Threshold Implications**

As the inspection milestone steps progress from $k=0$ out to $15$, the posterior density curves become significantly narrower and taller. This narrowing represents a substantial reduction in the variance of our estimate.

In terms of structural safety thresholds, this behavior is critical. Instead of relying on wide, uncertain confidence intervals that might delay maintenance or trigger false alarms, the sequential tracking framework rapidly builds a sharp, high-certainty estimation grid. This allows engineers to confidently detect when a structural element crosses a critical safety threshold (e.g., falling below 70% efficiency), enabling proactive maintenance before catastrophic failure occurs.